In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "src" / "utils" / "config.py").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Using project root: {project_root}")

Using project root: D:\image-processing-project


# Lib

In [ ]:
from pathlib import Path
import random
import joblib
import time

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm

from skimage.feature import hog

from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from src.utils.config import *
from src.utils.preprocess_detection import (
    read_yolo_label_file,
    box_iou
)

# Config

In [3]:
TRAIN_POSITIVE_DIR = TRAIN_DETECTION / "positive"
TRAIN_NEGATIVE_DIR = TRAIN_DETECTION / "negative"

VAL_POSITIVE_DIR = VAL_DETECTION / "positive"
VAL_NEGATIVE_DIR = VAL_DETECTION / "negative"

TEST_IMAGES_DIR = TEST_DETECTION / "images"
TEST_LABELS_DIR = TEST_DETECTION / "labels"

PATCH_SIZE = (64, 64)
MAX_TRAIN_PER_CLASS = None
MAX_VAL_PER_CLASS = None
FEATURE_CACHE_DIR = DETECTION_MODEL_DIR / "feature_cache"
MODEL_PATH = DETECTION_MODEL_DIR / "hog_svm_model.joblib"

HOG_PARAMS = {
    "orientations": 9,
    "pixels_per_cell": (8, 8),
    "cells_per_block": (2, 2),
    "block_norm": "L2-Hys",
    "transform_sqrt": True,
    "feature_vector": True
}

FEATURE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
DETECTION_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Function

## Load Images

In [4]:
def get_image_paths(folder):
    folder = Path(folder)

    image_paths = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp"]:
        image_paths.extend(folder.glob(ext))

    return sorted(image_paths)


train_positive_paths = get_image_paths(TRAIN_POSITIVE_DIR)
train_negative_paths = get_image_paths(TRAIN_NEGATIVE_DIR)

val_positive_paths = get_image_paths(VAL_POSITIVE_DIR)
val_negative_paths = get_image_paths(VAL_NEGATIVE_DIR)

print("Train positive:", len(train_positive_paths))
print("Train negative:", len(train_negative_paths))
print("Val positive:", len(val_positive_paths))
print("Val negative:", len(val_negative_paths))

Train positive: 200967
Train negative: 292704
Val positive: 24892
Val negative: 36088


## HOG Feature

In [5]:
def read_gray_patch(image_path, patch_size=(64, 64)):
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)

    if image is None:
        raise ValueError(f"Cannot read image: {image_path}")

    if image.shape[:2] != (patch_size[1], patch_size[0]):
        image = cv2.resize(
            image,
            patch_size,
            interpolation=cv2.INTER_AREA
        )

    return image


def extract_hog_feature(gray_patch):
    feature = hog(
        gray_patch,
        orientations=HOG_PARAMS["orientations"],
        pixels_per_cell=HOG_PARAMS["pixels_per_cell"],
        cells_per_block=HOG_PARAMS["cells_per_block"],
        block_norm=HOG_PARAMS["block_norm"],
        transform_sqrt=HOG_PARAMS["transform_sqrt"],
        feature_vector=HOG_PARAMS["feature_vector"]
    )

    return feature

In [6]:
def sample_paths(paths, max_count, seed=RANDOM_STATE):
    paths = list(paths)

    if max_count is None or len(paths) <= max_count:
        return paths

    rng = np.random.default_rng(seed)
    indices = rng.choice(len(paths), size=max_count, replace=False)

    return [paths[index] for index in sorted(indices)]


def get_hog_feature_length():
    dummy_patch = np.zeros((PATCH_SIZE[1], PATCH_SIZE[0]), dtype=np.uint8)

    return len(extract_hog_feature(dummy_patch))


def build_hog_dataset(
    positive_paths,
    negative_paths,
    dataset_name="dataset",
    max_per_class=None,
    use_cache=True
):
    positive_paths = sample_paths(
        positive_paths,
        max_count=max_per_class,
        seed=RANDOM_STATE
    )
    negative_paths = sample_paths(
        negative_paths,
        max_count=max_per_class,
        seed=RANDOM_STATE + 1
    )

    cache_name = f"{dataset_name}_hog_{len(positive_paths)}pos_{len(negative_paths)}neg.npz"
    cache_path = FEATURE_CACHE_DIR / cache_name

    if use_cache and cache_path.exists():
        cached = np.load(cache_path)
        print(f"Loaded cached {dataset_name} features: {cache_path}")

        return cached["X"], cached["y"]

    labeled_paths = [(path, 1) for path in positive_paths]
    labeled_paths += [(path, 0) for path in negative_paths]
    labeled_paths = shuffle(labeled_paths, random_state=RANDOM_STATE)

    feature_length = get_hog_feature_length()
    X = np.empty((len(labeled_paths), feature_length), dtype=np.float32)
    y = np.empty(len(labeled_paths), dtype=np.int32)

    write_index = 0
    failed_paths = []

    for image_path, label in tqdm(labeled_paths, desc=f"{dataset_name} HOG"):
        try:
            patch = read_gray_patch(image_path, patch_size=PATCH_SIZE)
            feature = extract_hog_feature(patch).astype(np.float32, copy=False)

            if feature.shape[0] != feature_length:
                raise ValueError(f"Unexpected HOG length: {feature.shape[0]}")

            X[write_index] = feature
            y[write_index] = label
            write_index += 1
        except Exception as error:
            failed_paths.append((str(image_path), str(error)))

    X = X[:write_index]
    y = y[:write_index]

    if failed_paths:
        print(f"Skipped {len(failed_paths)} unreadable/invalid patches.")

    if use_cache:
        np.savez_compressed(cache_path, X=X, y=y)
        print(f"Saved cached {dataset_name} features: {cache_path}")

    return X, y

In [ ]:
X_train, y_train = build_hog_dataset(
    train_positive_paths,
    train_negative_paths,
    max_per_class=MAX_TRAIN_PER_CLASS,
    dataset_name="train"
)

X_val, y_val = build_hog_dataset(
    val_positive_paths,
    val_negative_paths,
    max_per_class=MAX_VAL_PER_CLASS,
    dataset_name="val"
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

## HOG SVM

In [ ]:
hog_svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        max_iter=10000
    ))
])

start_time = time.time()

hog_svm_model.fit(X_train, y_train)

train_time = time.time() - start_time

print("Training completed.")
print("Training time:", round(train_time, 2), "seconds")

## Eval

In [ ]:
def evaluate(model, X, y, dataset_name):
    y_pred = model.predict(X)

    accuracy = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred, zero_division=0)
    recall = recall_score(y, y_pred, zero_division=0)
    f1 = f1_score(y, y_pred, zero_division=0)

    print(f"===== {dataset_name} Evaluation =====")
    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1-score :", round(f1, 4))
    print()
    print(classification_report(
        y,
        y_pred,
        target_names=["non-face", "face"],
        zero_division=0
    ))
    print("Confusion matrix:")
    print(confusion_matrix(y, y_pred))

    summary = {
        "dataset": dataset_name,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

    return summary

## Image pyramid + sliding window

In [ ]:
def image_pyramid(image, scale_factor=0.8, min_size=(64, 64)):
    current_image = image.copy()
    current_scale = 1.0

    yield current_scale, current_image

    while True:
        new_width = int(current_image.shape[1] * scale_factor)
        new_height = int(current_image.shape[0] * scale_factor)

        if new_width < min_size[0] or new_height < min_size[1]:
            break

        current_image = cv2.resize(
            current_image,
            (new_width, new_height),
            interpolation=cv2.INTER_AREA
        )

        current_scale *= scale_factor

        yield current_scale, current_image


def sliding_window(image, window_size=(64, 64), step_size=16):
    window_w, window_h = window_size

    for y in range(0, image.shape[0] - window_h + 1, step_size):
        for x in range(0, image.shape[1] - window_w + 1, step_size):
            window = image[y:y + window_h, x:x + window_w]

            yield x, y, window

## Non-Maximum Suppression

In [ ]:
def non_max_suppression(boxes, scores, iou_threshold=0.3):
    if len(boxes) == 0:
        return [], []

    boxes = np.array(boxes, dtype=np.float32)
    scores = np.array(scores, dtype=np.float32)

    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    areas = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
    order = scores.argsort()[::-1]

    keep = []

    while len(order) > 0:
        i = order[0]
        keep.append(i)

        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])

        w = np.maximum(0, xx2 - xx1)
        h = np.maximum(0, yy2 - yy1)

        intersection = w * h
        union = areas[i] + areas[order[1:]] - intersection

        iou = intersection / np.maximum(union, 1e-6)

        remaining = np.where(iou <= iou_threshold)[0]
        order = order[remaining + 1]

    final_boxes = boxes[keep].astype(int).tolist()
    final_scores = scores[keep].tolist()

    return final_boxes, final_scores

## Detect Faces

In [ ]:
def detect_faces_hog_svm(
    gray_image,
    score_threshold=0.8,
    scale_factor=0.8,
    step_size=16,
    nms_iou_threshold=0.3
):
    candidate_boxes = []
    candidate_scores = []

    start_time = time.time()

    for scale, resized_gray in image_pyramid(
        gray_image,
        scale_factor=scale_factor,
        min_size=PATCH_SIZE
    ):
        for x, y, window in sliding_window(
            resized_gray,
            window_size=PATCH_SIZE,
            step_size=step_size
        ):
            if window.shape[0] != PATCH_SIZE[1] or window.shape[1] != PATCH_SIZE[0]:
                continue

            feature = extract_hog_feature(window)
            feature = feature.reshape(1, -1)

            score = hog_svm_model.decision_function(feature)[0]

            if score >= score_threshold:
                x1 = int(x / scale)
                y1 = int(y / scale)
                x2 = int((x + PATCH_SIZE[0]) / scale)
                y2 = int((y + PATCH_SIZE[1]) / scale)

                candidate_boxes.append([x1, y1, x2, y2])
                candidate_scores.append(float(score))

    final_boxes, final_scores = non_max_suppression(
        candidate_boxes,
        candidate_scores,
        iou_threshold=nms_iou_threshold
    )

    elapsed_time = time.time() - start_time

    return final_boxes, final_scores, elapsed_time

## Draw Bbox

In [ ]:
def draw_boxes(image, boxes, color=(0, 255, 0), thickness=2, label=None):
    if len(image.shape) == 2:
        output = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    else:
        output = image.copy()

    for box in boxes:
        x1, y1, x2, y2 = box

        cv2.rectangle(
            output,
            (x1, y1),
            (x2, y2),
            color,
            thickness
        )

        if label is not None:
            cv2.putText(
                output,
                label,
                (x1, max(0, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2
            )

    return output

# Main

## Eval

In [ ]:
train_patch_summary = evaluate(
    hog_svm_model,
    X_train,
    y_train,
    dataset_name="Train"
)

val_patch_summary = evaluate(
    hog_svm_model,
    X_val,
    y_val,
    dataset_name="Validation"
)

patch_summary_df = pd.DataFrame([
    train_patch_summary,
    val_patch_summary
])

patch_summary_df

## Visualize

In [ ]:
test_image_paths = sorted(list(TEST_IMAGES_DIR.glob("*")))

sample_image_path = random.choice(test_image_paths)
sample_label_path = TEST_LABELS_DIR / f"{sample_image_path.stem}.txt"

gray_image = cv2.imread(str(sample_image_path), cv2.IMREAD_GRAYSCALE)

height, width = gray_image.shape[:2]

gt_boxes = read_yolo_label_file(
    label_path=sample_label_path,
    image_width=width,
    image_height=height
)

pred_boxes, scores, elapsed_time = detect_faces_hog_svm(
    gray_image=gray_image,
    score_threshold=0.8,
    scale_factor=0.8,
    step_size=16,
    nms_iou_threshold=0.3
)

gt_vis = draw_boxes(
    image=gray_image,
    boxes=gt_boxes,
    color=(255, 0, 0),
    label="GT"
)

pred_vis = draw_boxes(
    image=gray_image,
    boxes=pred_boxes,
    color=(0, 255, 0),
    label="HOG+SVM"
)

print("Image:", sample_image_path.name)
print("GT boxes:", len(gt_boxes))
print("Pred boxes:", len(pred_boxes))
print("Scores:", scores[:10])
print("Time:", round(elapsed_time, 4), "seconds")

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(gt_vis, cv2.COLOR_BGR2RGB))
plt.title("Ground Truth")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(pred_vis, cv2.COLOR_BGR2RGB))
plt.title("HOG + SVM Prediction")
plt.axis("off")

plt.tight_layout()
plt.show()

# Save

In [ ]:
model_data = {
    "model": hog_svm_model,
    "hog_params": HOG_PARAMS,
    "patch_size": PATCH_SIZE,
    "train_time": train_time,
    "train_patch_summary": train_patch_summary,
    "val_patch_summary": val_patch_summary
}

joblib.dump(model_data, MODEL_PATH)

print("Saved HOG + SVM model to:")
print(MODEL_PATH)